In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import time
import random
import os
import re
from datetime import datetime

# Initialize existing ID
existing_max_id = 4318  # fallback default
if os.path.exists("data/consumer_complaints.csv"):
    df_existing = pd.read_csv("data/consumer_complaints.csv")
    if "Unique ID" in df_existing.columns:
        ids = df_existing["Unique ID"].dropna().str.extract(r'(\d+)').astype(float)
        if not ids.empty:
            existing_max_id = int(ids.max().iloc[0])

next_id = existing_max_id + 1

BASE_URL = "https://indiankanoon.org"
SEARCH_URL = "https://indiankanoon.org/search/"
MAX_PAGES = 5  # pages per keyword, adjust as needed

OUTPUT_CSV = "data/indiankanoon_complaints.csv"
OUTPUT_JSON = "data/indiankanoon.json"
MASTER_CSV = "data/consumer_complaints.csv"   # existing master file
TXT_DIR = "data/txt"                           # same txt folder as consumer scraper
os.makedirs(TXT_DIR, exist_ok=True)
os.makedirs("data", exist_ok=True)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8"
}

In [2]:
KEYWORD_TAXONOMY = {
    "General Cybercrime / Cyber Fraud Terms": {
        "General Cybercrime": ["cyber crime", "cybercrime", "cyber fraud", "online fraud", "internet fraud", "digital fraud"],
        "Cyber Scam": ["cyber scam", "online scam", "internet scam", "online cheating"],
        "Financial Cyber Fraud": ["financial fraud online", "net banking fraud", "e-banking fraud"]
    },
    "UPI and Digital Payment Fraud": {
        "UPI Fraud": ["UPI fraud", "UPI scam", "Google Pay fraud", "PhonePe fraud", "Paytm fraud", "BHIM fraud"],
        "QR Code Fraud": ["QR code scam", "QR code fraud", "scan and pay fraud"],
        "Payment Link Fraud": ["payment link fraud", "collect request scam", "fake payment link"],
        "Mobile Wallet Fraud": ["mobile wallet fraud", "e-wallet scam", "digital wallet fraud"]
    },
    "OTP and Authentication Fraud": {
        "OTP Fraud": ["OTP fraud", "OTP scam", "OTP theft"],
        "SIM Swap": ["SIM swap fraud", "SIM cloning", "duplicate SIM fraud"],
        "KYC Fraud": ["KYC fraud", "KYC scam", "Aadhaar KYC scam"]
    },
    "Digital Arrest Scam": {
        "Digital Arrest": ["digital arrest", "digital arrest scam", "fake arrest", "video call arrest"],
        "Impersonation Scam": ["police impersonation scam", "CBI fraud call", "customs fraud call", "TRAI scam call", "ED scam call"],
        "Video Call Coercion": ["video call scam", "video call blackmail", "video call extortion", "fake interrogation"]
    },
    "Phishing, Vishing, and Smishing": {
        "Phishing": ["phishing", "phishing attack", "phishing email", "fake website", "spoof website"],
        "Vishing": ["vishing", "voice phishing", "fraud call", "fake bank call"],
        "Smishing": ["smishing", "SMS fraud", "SMS scam", "phishing SMS"]
    },
    "Online Lending and Loan App Fraud": {
        "Loan App Fraud": ["loan app fraud", "instant loan scam", "loan app harassment", "illegal loan app"],
        "Loan App Extortion": ["loan app blackmail", "loan app threat", "morphed photos loan", "recovery agent threat"]
    },
    "Investment and Trading Fraud": {
        "Investment Scam": ["investment scam", "Ponzi scheme", "online investment fraud", "crypto scam", "bitcoin fraud", "forex trading scam", "pig butchering"],
        "Stock Market Fraud": ["stock market scam", "share trading fraud", "demat fraud", "pump and dump"],
        "Task Scam": ["task fraud", "part time job scam", "work from home scam", "Telegram task scam"]
    },
    "Identity Theft and Data Breach": {
        "Identity Theft": ["identity theft", "identity fraud", "Aadhaar misuse", "PAN fraud"],
        "Data Breach": ["data breach", "data leak", "data theft", "customer data breach"]
    },
    "Social Engineering and Romance/Sextortion": {
        "Social Engineering": ["social engineering fraud", "manipulation scam", "trust scam"],
        "Romance Scam": ["romance scam", "dating fraud", "matrimonial fraud", "honey trap", "catfishing fraud"],
        "Sextortion": ["sextortion", "webcam blackmail", "nude video blackmail"]
    },
    "E-Commerce and Delivery Fraud": {
        "E-Commerce Fraud": ["e-commerce fraud", "online shopping fraud", "fake product scam", "Flipkart fraud", "Amazon fraud", "refund scam"],
        "Delivery Fraud": ["fake delivery", "courier fraud", "customs duty scam", "parcel scam", "delivery OTP scam"]
    },
    "Ransomware and Malware": {
        "Ransomware": ["ransomware attack", "cyber ransom", "data encryption attack"],
        "Banking Malware": ["banking trojan", "banking malware", "keylogger fraud", "AnyDesk fraud", "TeamViewer scam", "screen sharing scam"]
    },
    "Emerging and Miscellaneous Fraud Types": {
        "Deepfake Fraud": ["deepfake scam", "deepfake fraud", "AI voice scam", "voice cloning scam"],
        "Utility Scam": ["electricity bill scam", "utility fraud", "disconnection scam"],
        "Aadhaar Fraud": ["Aadhaar fraud", "Aadhaar scam", "biometric fraud", "AEPS fraud"],
        "Cyber Stalking": ["cyber stalking", "cyber bullying", "online harassment", "digital harassment"]
    }
}

In [3]:
def classify_narrative_type(text):
    text_lower = text.lower()
    if any(x in text_lower for x in ["complainant lost", "amount deducted", "cheated", "defrauded", "i lost", "victim"]):
        return "VICTIM"
    elif any(x in text_lower for x in ["attempted", "tried to", "almost", "suspicious", "did not share"]):
        return "NEAR-MISS"
    return "THIRD-PARTY"

def clean_text(text):
    """Remove excessive whitespace and normalize."""
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

In [4]:
# Initialize a Session for maintaining cookies across requests
if 'session' not in globals():
    session = requests.Session()
    # First visit homepage to get cookies
    try:
        session.get(BASE_URL, headers=HEADERS, timeout=15)
        time.sleep(2)
    except Exception as e:
        print(f"Error initializing session: {e}")

def search_indiankanoon(query, page=0):
    """Search Indian Kanoon and return list of result metadata dicts."""
    results = []
    seen_urls = set()

    params = {"formInput": query, "pagenum": page}
    try:
        time.sleep(random.uniform(2.0, 4.0))
        
        # Then search using the session
        response = session.get(SEARCH_URL, params=params, headers=HEADERS, timeout=15)
        
        if response.status_code != 200:
            print(f"  Failed: status {response.status_code}")
            return results

        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Indian Kanoon uses <article class="result"> for the search items
        result_items = soup.find_all('article', class_='result')

        # Fallback to div if article isn't used
        if not result_items:
            result_items = soup.find_all('div', class_='result')

        if not result_items:
            print(f"  No results found on page {page}.")
            return results

        for item in result_items:
            # Title and URL usually in <h4 class="result_title">
            title_tag = item.find('h4', class_='result_title')
            if not title_tag:
                title_tag = item.find('div', class_='title') # fallback
            
            if not title_tag:
                continue
                
            a_tag = title_tag.find('a')
            if not a_tag:
                continue

            title = a_tag.get_text(strip=True)
            href = a_tag.get('href', '')
            
            # The search results often link to /docfragment/, but we want the full /doc/ url to scrape text
            # e.g., /docfragment/88768514/?formInput=UPI%20fraud -> /doc/88768514/
            doc_id_match = re.search(r'/(?:doc|docfragment)/(\d+)', href)
            doc_id = doc_id_match.group(1) if doc_id_match else ""
            
            if doc_id:
                full_url = f"{BASE_URL}/doc/{doc_id}/"
            else:
                full_url = BASE_URL + href if href.startswith('/') else href

            if full_url in seen_urls:
                continue
            seen_urls.add(full_url)

            # Court and date from docsource & title
            docsource_span = item.find('span', class_='docsource')
            court = ""
            if docsource_span:
                court = docsource_span.get_text(separator=' ', strip=True)

            doc_date = "Unknown Date"
            # Date is often at the end of the title like "... on 17 October, 2022"
            date_match = re.search(r'on\s+(\d{1,2}\s+[A-Za-z]+\s*,\s*\d{4})', title)
            if date_match:
                doc_date = date_match.group(1)

            results.append({
                "doc_id": doc_id,
                "title": title,
                "url": full_url,
                "court": court,
                "date": doc_date,
            })

        print(f"  Found {len(results)} results on page {page}.")
        return results

    except Exception as e:
        print(f"  Error searching '{query}' page {page}: {e}")
        return results

In [5]:
def diagnose_indiankanoon(query="UPI fraud"):
    """
    Diagnostic tool to peek at the raw HTML structure of Indian Kanoon.
    Used to verify CSS selectors for scraping.
    """
    params = {"formInput": query, "pagenum": 0}
    response = requests.get(SEARCH_URL, params=params, headers=HEADERS, timeout=15)
    print(f"Status: {response.status_code}")
    soup = BeautifulSoup(response.text, 'html.parser')

    # Print ALL unique div classes on the page
    print("\n=== All div classes found ===")
    all_classes = set()
    for tag in soup.find_all('div', class_=True):
        for c in tag.get('class', []):
            all_classes.add(c)
    for c in sorted(all_classes):
        print(f"  .{c}")

    # Print first 3000 chars of raw HTML to see structure
    print("\n=== Raw HTML (first 3000 chars) ===")
    print(response.text[:3000])

# uncomment to run
# diagnose_indiankanoon()

In [6]:
def fetch_judgment_text(url, doc_id):
    """Scrape full judgment text from a specific document page."""
    try:
        time.sleep(random.uniform(5.0, 10.0))
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/121.0.0.0 Safari/537.36",
            "Referer": SEARCH_URL
        }
        # Using the global session to persist cookies
        resp = session.get(url, headers=headers, timeout=20)
        
        if resp.status_code == 404:
            return None, "404 Not Found"
            
        if "captcha" in resp.url.lower() or resp.status_code == 403:
            return None, "BLOCKED BY CAPTCHA OR 403"
            
        if resp.status_code != 200:
            return None, f"Status code {resp.status_code}"

        soup = BeautifulSoup(resp.text, 'html.parser')
        
        # Indian Kanoon usually puts judgment text in div with class "judgments"
        judgment_div = soup.find('div', class_='judgments')
        if not judgment_div:
            return None, "No text div found ('judgments' class missing)"

        # Remove header, footnotes, or extraneous spans if needed
        # Often paragraphs are in <p id="p_1"> ... </p>
        paragraphs = judgment_div.find_all('p')
        if not paragraphs:
            # Fallback to pure text
            text = clean_text(judgment_div.get_text(separator=' ', strip=True))
        else:
            text_lines = [clean_text(p.get_text(separator=' ', strip=True)) for p in paragraphs]
            text = " ".join([t for t in text_lines if t])

        # Return cleaned full narrative text
        return text, "Success"

    except requests.exceptions.Timeout:
        return None, "Timeout"
    except Exception as e:
        return None, f"Error: {e}"

In [ ]:
import os

def safe_save(df, csv_path, json_path, json_data):
    """Save to temp file first, then rename — avoids PermissionError if Excel has file open."""
    try:
        # Save CSV via temp file
        temp_csv = csv_path + ".tmp"
        df.to_csv(temp_csv, index=False, encoding="utf-8-sig")
        os.replace(temp_csv, csv_path)

        # Save JSON via temp file
        temp_json = json_path + ".tmp"
        with open(temp_json, "w", encoding="utf-8") as f:
            json.dump(json_data, f, indent=4, ensure_ascii=False)
        os.replace(temp_json, json_path)

        print(f"✅ Checkpoint saved: {len(df)} records")
    except Exception as e:
        print(f"⚠️ Checkpoint save failed (data still in memory): {e}")

# --- MAIN SCRAPING LOOP FOR INDIAN KANOON ---
# Load existing JSON if available to append and avoid duplicates
try:
    with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
        master_data_json = json.load(f)
        scraped_doc_ids = set()
        for item in master_data_json:
            notes = item.get('Notes', '')
            if 'Doc ID: ' in notes:
                doc_id = notes.split('Doc ID: ')[-1]
                scraped_doc_ids.add(doc_id)
        print(f"Loaded {len(scraped_doc_ids)} existing scraped documents from JSON.")
except (FileNotFoundError, json.JSONDecodeError):
    master_data_json = []
    scraped_doc_ids = set()
    print("No existing JSON database found, starting fresh.")

new_results_count = 0
MAX_PAGES_PER_QUERY = 10 

for parent_category, subcategories in KEYWORD_TAXONOMY.items():
    print(f"\n--- Processing Category: {parent_category} ---")
    
    for subcat, keyword_list in subcategories.items():
        for word in keyword_list:
            print(f"\n  Keyword: '{word}'")
            for page_num in range(0, MAX_PAGES_PER_QUERY):
                search_results = search_indiankanoon(word, page=page_num)
                
                if not search_results:
                    break # Stop paginating if empty
                    
                for res in search_results:
                    doc_id = res['doc_id']
                    if not doc_id:
                         continue
                         
                    if doc_id in scraped_doc_ids:
                        print(f"    Skipping already scraped doc {doc_id}")
                        continue

                    print(f"    -> Fetching doc {doc_id}: {res['title'][:50]}...")
                    judgment_text, status = fetch_judgment_text(res['url'], doc_id)

                    if judgment_text:
                        # Increment the global existing Unique ID to match the CSV architecture exactly
                        existing_max_id += 1
                        assigned_id = f"NA-{existing_max_id:04d}"
                        
                        txt_filename = f"{assigned_id}.txt"
                        with open(os.path.join("data/txt", txt_filename), "w", encoding="utf-8") as f_txt:
                            f_txt.write(judgment_text)
                        
                        data_row = {
                            "Unique ID": assigned_id,
                            "Date of Collection": datetime.now().strftime("%Y-%m-%d"),
                            "Collector Name": "Soubhik Sarkar",
                            "Source Platform": "Indian Kanoon",
                            "Source Publication": "indiankanoon.org",
                            "Original Date": res['date'],
                            "Title/Headline": res['title'],
                            "URL": res['url'],
                            "Search Query Used": word,
                            "Fraud Category": parent_category,
                            "Fraud Subcategory": subcat,
                            "Narrative Type": classify_narrative_type(judgment_text),
                            "TXT File Name": txt_filename,
                            "Notes": f"Court: {res['court']} | Doc ID: {doc_id}"
                        }
                        
                        master_data_json.append(data_row)
                        scraped_doc_ids.add(doc_id)
                        new_results_count += 1
                        print(f"       ✅ Saved ID: {assigned_id}. Length: {len(judgment_text)}")

                        # Save intermediate every 25 new rows
                        if new_results_count % 25 == 0:
                            df_temp = pd.DataFrame(master_data_json)
                            safe_save(df_temp, OUTPUT_CSV, OUTPUT_JSON, master_data_json)
                            
                    else:
                        print(f"       ❌ Failed doc {doc_id}: {status}")
                        
            print(f"  Finished parsing pages for '{word}'.")
        
print(f"\nTotal new judgements scraped this session: {new_results_count}!")

No existing JSON database found, starting fresh.

--- Processing Category: General Cybercrime / Cyber Fraud Terms ---

  Keyword: 'cyber crime'
  Found 10 results on page 0.
    -> Fetching doc 139318141: Kotak Mahindra Bank Ltd. vs K.Seetharam Bhat on 14...
       ✅ Saved ID: NA-4319. Length: 28200
    -> Fetching doc 157003932: Mahesh Kumar Poddar vs The State Of Jharkhand on 1...
       ✅ Saved ID: NA-4320. Length: 16192
    -> Fetching doc 147390199: The Chief Manager vs K.Seetharam Bhat on 14 Januar...
       ✅ Saved ID: NA-4321. Length: 28200
    -> Fetching doc 136280608: The Chief Manager vs K.Seetharam Bhat on 14 Januar...
       ✅ Saved ID: NA-4322. Length: 28200
    -> Fetching doc 56699948: Jaydeep Madhukar Wakankar vs The State Of Andhra P...
       ✅ Saved ID: NA-4323. Length: 44827
    -> Fetching doc 68207149: Adv R.Mahalakshmi vs Commissioner Of Police on 21 ...
       ✅ Saved ID: NA-4324. Length: 15800
    -> Fetching doc 71199828: K. Mathamma vs The State Of Telangan

PermissionError: [Errno 13] Permission denied: 'data/indiankanoon_complaints.csv'

In [ ]:
# FINAL SAVING ROUTINE
if new_results_count > 0:
    import pandas as pd
    import json
    import os

    try:
        os.makedirs('data', exist_ok=True)
    except Exception as e:
        print(f"Error creating data folder: {e}")

    df_final = pd.DataFrame(master_data_json)
    
    df_final.to_csv(OUTPUT_CSV, index=False)
    print(f"✅ Saved CSV to {OUTPUT_CSV} with {len(df_final)} total records.")

    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
        json.dump(master_data_json, f, indent=4)
    print(f"✅ Saved JSON to {OUTPUT_JSON}")
else:
    print("No new data to save.")

# Quick summary view
if 'df_final' in locals():
    display(df_final.tail(2))

In [14]:
# === DIAGNOSTIC / STANDALONE CELL ===
# Use this to test a SINGLE indian kanoon search query and URL text fetch directly 
# without triggering the main loop and affecting the master datasets.

test_query = "flipkart fraud"
print(f"Testing search for: '{test_query}'")

res = search_indiankanoon(test_query, page=0)
if res:
    first_hit = res[0]
    print(json.dumps(first_hit, indent=2))
    
    print("\nAttempting text scrape for this document...")
    txt_content, stat = fetch_judgment_text(first_hit['url'], first_hit['doc_id'])
    
    if txt_content:
        print(f"Scrape Status: {stat}")
        print("-" * 40)
        # Print first 500 characters
        print(txt_content[:500] + "...\n[TRUNCATED DIAGNOSTIC VIEW]")
    else:
        print(f"Failed to fetch text. Error: {stat}")
else:
    print("Search returned no results.")

Testing search for: 'flipkart fraud'
  Found 10 results on page 0.
{
  "doc_id": "149811803",
  "title": "Mirtunaj Kumar vs The Additional Chief Secretary To ... on 29 April, 2025",
  "url": "https://indiankanoon.org/doc/149811803/",
  "court": "Madras High Court",
  "date": "29 April, 2025"
}

Attempting text scrape for this document...
Scrape Status: Success
----------------------------------------
DR. G.JAYACHANDRAN, J. AND
 R.POORNIMA, J. This Habeas Corpus Petition is filed by the accused in

 multiple criminal cases of financial fraud, who has been detained by the

 second respondent in Proceedings No. 52 of 2024, dated 04.09.2024,

 under Act 14 of 1982. 2. Heard on either side and perused the material documents

 available on record. 3. This Habeas Corpus Petition is filed on the ground that

 the Detaining Authority has not followed the prescribed procedures while 2/7 https://www.mh...
[TRUNCATED DIAGNOSTIC VIEW]


In [8]:
print(f"Records in memory: {len(master_data_json)}")
print(f"New records this session: {new_results_count}")

Records in memory: 3175
New records this session: 3175
